In [0]:
#%run ./translation_function ----- A décommmenter pour lancer les notebooks séparements

# build_nomenclatures

Construit les **dimensions de nomenclature** : une ligne par entité métier,
avec son code technique.

## Pourquoi elles sont nécessaires

Sans elles, la relation Power BI entre les faits et une table de traduction est
**plusieurs-à-plusieurs** : `dim_batches_specifications[id_good_variety]` n'est
pas unique (plusieurs batches par variété) et `dim_trad_variety[id_good_variety]`
non plus (une ligne par langue). Deux côtés « plusieurs ».

La nomenclature rétablit un schéma en étoile classique :

```
dim_batches_specifications  --*→1--  dim_variety  --1←*--  dim_trad_variety
                                   (1 ligne/variété)      (4 lignes/variété)
```

Chaque relation a un côté « 1 » identifié. Le modèle devient lisible, et le
`code` de la nomenclature sert de **colonne de tri** : l'ordre des libellés reste
stable d'une langue à l'autre.

Ces dimensions dépassent le seul besoin de traduction — une `dim_variety` avec
son code et ses attributs est utile en soi, pour ce projet comme pour les autres.

In [0]:
# Sources : les tables métier de la base PostgreSQL du front.
goods_species = spark.table(f"{source_catalog}.goods_species")
goods_varieties = spark.table(f"{source_catalog}.goods_varieties")
requirement_specifications = spark.table(f"{source_catalog}.requirement_specifications")
parameters_production_types = spark.table(f"{source_catalog}.parameters_production_types")
parameters_variables = spark.table(f"{source_catalog}.parameters_variables")

## Les nomenclatures

Toutes filtrées sur `deleted = false`, comme le reste du pipeline.

Le `code` est conservé : c'est le libellé technique, non traduit, qui servira de
colonne de tri dans Power BI et de repère pour le debug.

In [0]:
# goods_species -> dim_specy
dim_specy = (
    goods_species
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_good_specy"),
        F.col("code").alias("specy_code"),
    )
)

publish_dim(dim_specy, "dim_specy", ["id_good_specy"], translations_schema,
            is_translation=False)

In [0]:
# goods_varieties -> dim_variety
# La variété porte sa propre espèce (colonne specy) : on la garde, elle permet
# une hiérarchie espèce > variété dans le modèle.
dim_variety = (
    goods_varieties
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_good_variety"),
        F.col("code").alias("variety_code"),
        F.col("specy").alias("id_good_specy"),
    )
)

publish_dim(dim_variety, "dim_variety", ["id_good_variety"], translations_schema,
            is_translation=False)

In [0]:
# parameters_production_types -> dim_production_type
dim_production_type = (
    parameters_production_types
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_production_type"),
        F.col("code").alias("production_type_code"),
    )
)

publish_dim(dim_production_type, "dim_production_type",
            ["id_parameter_production_type"], translations_schema,
            is_translation=False)

In [0]:
# parameters_variables -> dim_variable
dim_variable = (
    parameters_variables
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_parameter_variable"),
        F.col("code").alias("variable_code"),
    )
)

publish_dim(dim_variable, "dim_variable", ["id_parameter_variable"],
            translations_schema, is_translation=False)

## `requirement_specifications` — sans table de traduction

Cette nomenclature est construite pour compléter le modèle (relation propre,
colonne de tri), mais **il n'existe pas de `requirement_specifications_translations`
en base**. Son libellé restera donc dans la langue de saisie.

À confirmer avec le PO : oubli du front, ou champ volontairement non traduit ?

In [0]:
# requirement_specifications -> dim_requirement_specification
dim_requirement_specification = (
    requirement_specifications
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_requirement_specification"),
        F.col("name").alias("requirement_specification_name"),
    )
)

publish_dim(dim_requirement_specification, "dim_requirement_specification",
            ["id_requirement_specification"], translations_schema,
            is_translation=False)

## Nomenclatures restant à construire

Quatre tables de traduction n'ont pas encore leur nomenclature, faute de
connaître la table métier source et ses colonnes :

| Traduction | Table métier source |
|---|---|
| `dim_trad_localization` | ? |
| `dim_trad_localization_group` | ? |
| `dim_trad_production_line_variable` | ? |
| `dim_trad_batch_note_*` | ? |

En attendant, ces quatre-là garderont une relation plusieurs-à-plusieurs avec
les faits — fonctionnel, mais moins propre que le reste.

> **Solution de repli si la table métier n'existe pas** : dériver la nomenclature
> des clés distinctes de la table de traduction elle-même. Cela donne bien un
> côté « 1 », mais reproduit la limite connue — une clé absente de toute
> traduction resterait invisible.